# Midas Design Guide — Prestressed Concrete Bridge Load Rating

**Companion notebook** for the corresponding chapter of the MIDAS training
manual *Design Guide for midas Civil — AASHTO LRFD*. The guide itself is
proprietary and is **not reproduced here** — this notebook contains only
original code and AASHTO LRFD / MBE article citations.

**How this notebook is used.** Work through the design process with this
notebook alongside: each step carries its AASHTO background in original
words, a live Python environment for exploratory checks and quick
validation, a direct interface to Midas Civil through its API, and
customization to ODOT's design process (PSID / PSBD standard products,
ODOT materials and vehicles).
Everything the guide has you do by hand in the Midas UI — coordinate entry,
side math — is demonstrated here as runnable code a designer can follow,
rerun, and modify. Where a number must come from the
Midas model itself, it is either pulled live over the Civil NX JSON API
(the verified result tables run in place) or recorded from the companion
six-beam model with its provenance noted; the remaining `TODO(midas-api)`
markers await the PSC design module and construction-stage analysis. Hand-entered values (from a standard
drawing or a hand calc) are compared via the `check()` harness below.

**Scope of this chapter:**

1. Rating vehicle set and MBE 6A load/resistance factors
2. Bridge inputs (usually the Chapter 1 girder) and capacities at rating limit states
3. Force effects per vehicle from the Midas moving-load analysis (API)
4. Design load rating — HL-93 inventory / operating (strength + Service III)
5. Legal load rating with ADTT-based factors
6. Permit load rating
7. Posting evaluation and cross-check vs Midas rating tables (API)


In [1]:
import math
import pandas as pd

# civilpy is installed editable into this env (pip install -e .)
from civilpy.structural.midas import MidasCivil, parse_result_table, envelope
from civilpy.structural.aashto.lrfd import (
    concrete, prestressed, steel, composite, distribution, lrfr, creep_shrinkage,
)

# --- Midas Civil NX connection -------------------------------------------------
# The API is not running on this machine right now. Everything below that needs
# the live model is guarded by MIDAS_ONLINE and marked TODO(midas-api).
try:
    midas = MidasCivil()
    MIDAS_ONLINE = midas.ping()
except Exception:
    midas, MIDAS_ONLINE = None, False
print("Midas Civil NX online:", MIDAS_ONLINE)

Midas Civil NX online: True


In [2]:
# --- Validation harness --------------------------------------------------------
# Every comparison in this notebook goes through check() so the end-of-notebook
# summary shows guide value vs civilpy value side by side.
RESULTS = []

def check(label, guide_value, civilpy_value, tol=0.01, unit=""):
    """Compare a guide-reported value against the civilpy-computed one.

    tol is relative (1% default) — the guide rounds intermediate values, so
    small drift is expected; flag anything beyond tol for investigation.
    """
    if guide_value is None or civilpy_value is None:
        status = "PENDING"
        diff = None
    else:
        diff = abs(civilpy_value - guide_value) / (abs(guide_value) or 1.0)
        status = "OK" if diff <= tol else "MISMATCH"
    RESULTS.append({"check": label, "guide": guide_value, "civilpy": civilpy_value,
                    "rel diff": diff, "unit": unit, "status": status})
    print(f"[{status}] {label}: guide={guide_value} civilpy={civilpy_value} {unit}")
    return status == "OK"

def summary():
    df = pd.DataFrame(RESULTS)
    if len(df):
        n_ok = (df.status == "OK").sum()
        print(f"{n_ok}/{len(df)} checks OK, "
              f"{(df.status == 'MISMATCH').sum()} mismatches, "
              f"{(df.status == 'PENDING').sum()} pending")
    return df

## 1. Rating setup — MBE Part 6A

Vehicles, condition/system factors, and the limit states rated. PSC design
load rating checks Strength I plus the Service III stress limit (inventory).

### 1.1 ODOT rating vehicle set — BDM Section 908.3

ODOT does not rate for the AASHTO legal loads alone. BDM 908.3 requires
**every bridge** to be rated for ten commercial legal vehicles and two
emergency vehicles, and ODOT bridges additionally for two state permit
loads by policy:

| group | vehicles | BDM figure |
|---|---|---|
| Ohio commercial legal | S-2F1, S-3F1, S-5C1 | 908.3-1 |
| AASHTO legal | Type 3, Type 3S2, Type 3-3 | 908.3-2 |
| Specialized hauling (SHV) | SU4, SU5, SU6, SU7 | 908.3-3 |
| Emergency (FAST Act) | EV2, EV3 | 908.3-4 |
| ODOT state permit | S-PL60T, S-PL65T | 908.3-5 |

Inventory and operating ratings use the design loads of BDM 908.2 —
HS20 (truck *or* lane, whichever governs) and HL-93.

Two vehicles Midas offers are deliberately **not** in this set: the
**AASHTO National Rating Load**, because ODOT rates SU4–SU7 individually
rather than using the NRL envelope, and **Ohio 4F1**, which exists in
Midas's OHDOT DB but is absent from the 908.3 list.

`civilpy.structural.aashto.vehicles` carries the axle trains, each read
off the BDM figure and checked below.

In [3]:
from civilpy.structural.aashto.vehicles import (
    BDM_908_RATING_LOADS, RATING_VEHICLES)

# GVW and overall length as printed on BDM Figures 908.3-1 .. 908.3-5.
BDM_FIG = {
    "2F1": (30.0, 10.0), "3F1": (46.0, 14.0), "5C1": (80.0, 51.0),
    "Type 3": (50.0, 19.0), "Type 3S2": (72.0, 41.0), "Type 3-3": (80.0, 54.0),
    "SU4": (54.0, 18.0), "SU5": (62.0, 22.0), "SU6": (69.5, 26.0),
    "SU7": (77.5, 30.0), "EV2": (57.5, 15.0), "EV3": (86.0, 19.0),
    "S-PL60T": (120.0, 65.083), "S-PL65T": (130.0, 48.0),
}

rows = []
for name in BDM_908_RATING_LOADS:
    v = RATING_VEHICLES[name]
    gvw_fig, len_fig = BDM_FIG[name]
    check(f"{name} GVW", gvw_fig, v.gvw_kip, unit="kip")
    rows.append({
        "vehicle": name,
        "axles": len(v.axle_loads_kip),
        "axle loads (kip)": ", ".join(f"{p:g}" for p in v.axle_loads_kip),
        "spacings (ft)": ", ".join(f"{s:g}" for s in v.axle_spacings_ft),
        "GVW (kip)": v.gvw_kip,
        "length (ft)": v.wheelbase_ft,
    })
display(pd.DataFrame(rows).set_index("vehicle"))

[OK] 2F1 GVW: guide=30.0 civilpy=30.0 kip
[OK] 3F1 GVW: guide=46.0 civilpy=46.0 kip
[OK] 5C1 GVW: guide=80.0 civilpy=80.0 kip
[OK] Type 3 GVW: guide=50.0 civilpy=50.0 kip
[OK] Type 3S2 GVW: guide=72.0 civilpy=72.0 kip
[OK] Type 3-3 GVW: guide=80.0 civilpy=80.0 kip
[OK] SU4 GVW: guide=54.0 civilpy=54.0 kip
[OK] SU5 GVW: guide=62.0 civilpy=62.0 kip
[OK] SU6 GVW: guide=69.5 civilpy=69.5 kip
[OK] SU7 GVW: guide=77.5 civilpy=77.5 kip
[OK] EV2 GVW: guide=57.5 civilpy=57.5 kip
[OK] EV3 GVW: guide=86.0 civilpy=86.0 kip
[OK] S-PL60T GVW: guide=120.0 civilpy=120.0 kip
[OK] S-PL65T GVW: guide=130.0 civilpy=130.0 kip


,axles,axle loads (kip),spacings (ft),GVW (kip),length (ft)
vehicle,,,,,
2F1,2,"10, 20",10,30.0,10.000
3F1,3,"12, 17, 17","10, 4",46.0,14.000
5C1,5,"12, 17, 17, 17, 17","12, 4, 31, 4",80.0,51.000
Type 3,3,"16, 17, 17","15, 4",50.0,19.000
Type 3S2,5,"10, 15.5, 15.5, 15.5, 15.5","11, 4, 22, 4",72.0,41.000
Type 3-3,6,"12, 12, 12, 16, 14, 14","15, 4, 15, 16, 4",80.0,54.000
SU4,4,"12, 8, 17, 17","10, 4, 4",54.0,18.000
SU5,5,"12, 8, 8, 17, 17","10, 4, 4, 4",62.0,22.000
SU6,6,"11.5, 8, 8, 17, 17, 8","10, 4, 4, 4, 4",69.5,26.000


### 1.2 Load and resistance factors — BDM 924

**Dynamic load allowance (BDM 924.4).** 33% for all non-buried bridges,
adopting LRFD Table 3.6.2.1-1 — but the article carries five conditions a
blanket 33% gets wrong:

| clause | rule |
|---|---|
| A | 33% for all non-buried bridges, except fatigue |
| B | **15%** for fatigue evaluation |
| C | IM on the **truck or tandem** portion of HL-93 only — **not the lane portion** |
| D | **No IM on wood** components |
| E | IM **may be ignored** for slow-moving (< 10 mph), special or permit loads under controlled conditions |
| F | Buried: $IM = 33\,(1 - 0.125\,D_E) \ge 0\%$, $D_E$ = min cover (ft) |

**Permit live-load factors (BDM 924.3).** $\gamma_{DC}=1.25$,
$\gamma_{DW}=1.50$ at Strength II; $\gamma_{LL}=1.40$ for routine permits
(unlimited crossings) and $1.20$ for single-trip/limited-crossing permit
or special loads.

**EV3 is factored differently from every other legal load** —
$\gamma_{LL}=1.20$ on the Interstate system, $1.10$ elsewhere
(924.3 footnote), rather than the MBE legal-load factor.

In [4]:
IM_BDM = {
    "standard": 33.0,        # 924.4.A — non-buried, non-fatigue
    "fatigue": 15.0,         # 924.4.B
    "lane_portion": 0.0,     # 924.4.C — no IM on the lane load
    "wood": 0.0,             # 924.4.D
    "permit_controlled": 0.0,  # 924.4.E — may be ignored
}

def im_buried_percent(cover_ft):
    """BDM 924.4.F / LRFD 3.6.2.2-1 — IM reduced by depth of cover."""
    return max(0.0, 33.0 * (1.0 - 0.125 * cover_ft))

def gamma_ll_ev3(interstate):
    """BDM 924.3 footnote — EV3 carries its own live-load factor."""
    return 1.20 if interstate else 1.10

for cover, expect in ((0.0, 33.0), (4.0, 16.5), (8.0, 0.0)):
    print(f"  cover {cover:g} ft -> IM {im_buried_percent(cover):.1f}%")
check("EV3 gamma_LL, Interstate", 1.20, gamma_ll_ev3(True))
check("EV3 gamma_LL, non-Interstate", 1.10, gamma_ll_ev3(False))

  cover 0 ft -> IM 33.0%
  cover 4 ft -> IM 16.5%
  cover 8 ft -> IM 0.0%
[OK] EV3 gamma_LL, Interstate: guide=1.2 civilpy=1.2 
[OK] EV3 gamma_LL, non-Interstate: guide=1.1 civilpy=1.1 


True

In [5]:
# TODO(guide): record the guide's factor table here:
GUIDE = dict(
    phi_c=None,   # condition factor
    phi_s=None,   # system factor
    gamma_DC=1.25, gamma_DW=1.50,
    gamma_LL_inv=1.75, gamma_LL_op=1.35,
    adtt=None,
)
# Ohio legal vehicles already live in civilpy.structural.aashto.vehicles;
# TODO(guide): confirm which legal/permit vehicles the guide rates.
GUIDE

{'phi_c': None,
 'phi_s': None,
 'gamma_DC': 1.25,
 'gamma_DW': 1.5,
 'gamma_LL_inv': 1.75,
 'gamma_LL_op': 1.35,
 'adtt': None}

## 2. Capacities at the rating limit states

Reuse the Chapter 1 design notebook's capacity results (flexure, shear,
Service III stress margin). If rating a different example bridge, rebuild the
capacities here with the same civilpy calls.

In [6]:
# TODO(guide): phi*Mn, phi*Vn at the rated sections, and the Service III
# allowable-stress capacity term for the stress-based rating.
pass

## 3. Force effects per rating vehicle

In [7]:
if MIDAS_ONLINE:
    # Verified vs live Civil NX 2026-07-27: /post/TABLE selects by
    # TABLE_TYPE ("BEAMFORCE"); TABLE_NAME is just a label.
    try:
        resp = midas.result_table("Moving load envelopes",
                                  table_type="BEAMFORCE")
        display(pd.DataFrame(parse_result_table(resp)).head())
    except Exception as err:
        # a fresh/unanalyzed session (or a DB edit, or a pre-mode view
        # switch) clears results — analyze and rerun this cell
        print("no results in the session:", str(err)[-80:])
else:
    print("Midas offline — skipping Moving load envelopes (BEAMFORCE)")

no results in the session: ng load envelopes] Cannot generate table data as there is no analysis result.'}}


### 3.1 Loading the rating vehicles into Midas

`load_oh_vehicles` pushes exactly the BDM Section 908 set — nothing more —
and applies the 924.4 rules that can live on the vehicle record: lane
loads and the two state permit loads are written with IM = 0, everything
else with 33%.

Three Midas behaviours make this worth doing from code rather than by
hand, all verified against a live Civil NX:

1. **A standard-DB vehicle hides its axle data.** The stored record
   carries only `VEHICLE_TYPE_NAME` and `STANDARD_CODE`, so neither the
   API nor an export will tell you what axle train Midas applies — it has
   to be read off the UI dialog and compared against the table in §1.1.
   Midas's built-ins are a vendor transcription frozen at some past spec
   revision (its ODOT standard box sections are demonstrably out of date).
2. **An unresolved vehicle name fails silently.** A deliberately invalid
   name stores exactly as cleanly as a valid one and then applies **zero
   load**, with no error anywhere. Always confirm non-zero envelopes after
   the first analysis.
3. **`DYN_LOAD_ALLOWANCE` is discarded on user-defined vehicles.** It
   stores on standard-DB records only. With `use_standard_db=False` the
   allowance must be applied downstream in the moving-load case or the
   combination.

In [8]:
from civilpy.structural.midas_models import load_oh_vehicles

if MIDAS_ONLINE:
    out = load_oh_vehicles(midas, use_standard_db=True, replace=True,
                           im_percent=IM_BDM["standard"])
    stored = midas.request("GET", "db/mvhl").get("MVHL", {})
    display(pd.DataFrame([
        {"id": int(k),
         "vehicle": v.get("VEHICLE_LOAD_NAME"),
         "source": v.get("STANDARD_CODE") or "user-defined",
         "IM %": v.get("VEH_DEFAULT", {}).get("DYN_LOAD_ALLOWANCE", 0)}
        for k, v in stored.items()]).sort_values("id").set_index("id"))
    print("user-defined (no DB entry exists):", out["user_defined"])
else:
    print("Midas offline — load_oh_vehicles(midas, use_standard_db=True, "
          "replace=True) pushes the 908 set")

,vehicle,source,IM %
id,,,
1,HL-93 truck,AASHTO-LRFD,33
2,HL-93 tandem,AASHTO-LRFD,33
3,HS20-44 truck,AASHTO-STD,33
4,HS20-44 lane,AASHTO-STD,0
5,2F1,OHDOT LOAD,33
6,3F1,OHDOT LOAD,33
7,5C1,OHDOT LOAD,33
8,Type 3,AASHTO LEGAL/PERMIT LOAD,33
9,Type 3S2,AASHTO LEGAL/PERMIT LOAD,33


user-defined (no DB entry exists): ['S-PL60T', 'S-PL65T']


## 4. Design load rating — HL-93

RF = (C − γDC·DC − γDW·DW) / (γLL·(LL+IM)) at Strength I, plus the
Service III stress-based rating for inventory.

In [9]:
# TODO(guide):
# rf_inv = lrfr.rating_factor(capacity=..., dc=..., dw=..., ll_im=...,
#                             gamma_ll=1.75, phi_c=..., phi_s=...)
# rf_op  = lrfr.rating_factor(..., gamma_ll=1.35)
# check("RF inventory (flexure)", <guide>, rf_inv), etc.
pass

## 5. Legal load rating — MBE 6A.4.4

Generalized live-load factor from ADTT via `lrfr.legal_load_factor`.

In [10]:
# TODO(guide):
# g_ll = lrfr.legal_load_factor(adtt=GUIDE["adtt"])
# RF per legal vehicle; posting check below if any RF < 1.0.
pass

### Emergency vehicles — EV3 is not just another legal load

The EV3 **axle train is the unmodified FAST Act definition**, but ODOT
treats it differently from every other legal load in four ways. Batching
it with the commercial vehicles is the easy mistake:

- **Interstate only.** The BDM's definition of *Ohio Legal Vehicles*
  reads "…and emergency vehicle EV2; EV3 is legal to operate on
  Interstate system." EV2 is legal everywhere; EV3 is not.
- **Its own live-load factor** — $\gamma_{LL}=1.20$ Interstate / $1.10$
  elsewhere (924.3), not the MBE legal-load factor.
- **Its own lane placement** (916.D) — a single EV3 in one lane with the
  heaviest *commercial* legal loads in the remaining lanes, never a
  second EV3.
- **Its own posting threshold** (919.1) — posting triggers at
  $RF < 1.00$ for emergency vehicles, against $1.08$ for commercial
  legal loads.

In [11]:
POSTING_THRESHOLD = {"commercial_legal": 1.08, "emergency": 1.00}  # BDM 919.1

def needs_posting(rating_factor, vehicle):
    """BDM 919.1 — posting thresholds differ by vehicle class."""
    key = "emergency" if vehicle in ("EV2", "EV3") else "commercial_legal"
    return rating_factor < POSTING_THRESHOLD[key]

# an RF of 1.05 posts a commercial legal load but not an EV
check("posting threshold, commercial legal", 1.08,
      POSTING_THRESHOLD["commercial_legal"])
check("posting threshold, emergency", 1.00, POSTING_THRESHOLD["emergency"])
assert needs_posting(1.05, "Type 3") and not needs_posting(1.05, "EV3")
print("RF 1.05 -> post Type 3:", needs_posting(1.05, "Type 3"),
      "| post EV3:", needs_posting(1.05, "EV3"))

[OK] posting threshold, commercial legal: guide=1.08 civilpy=1.08 
[OK] posting threshold, emergency: guide=1.0 civilpy=1.0 
RF 1.05 -> post Type 3: True | post EV3: False


### Long-span and negative-moment provisions

Two ODOT rules change the *loading arrangement* rather than the vehicle:

- **Negative moment and interior reactions** (908.3): a 0.2 klf lane load
  with **two Type 3-3 vehicles at 0.75**, same direction, 30 ft apart —
  the larger of that and the legal loads applied separately governs.
  BDM 918.2.3.4 extends the same combination to all spans over 200 ft.
- **Ohio 5C1 trains** (918.2.3.1–.3): on long spans the right-most lane
  carries a *series* of 5C1 vehicles spaced 30 ft rear-axle to
  front-axle, as many as produce the maximum effect, with **no partial
  vehicles**; other lanes carry single 5C1s.

In [12]:
NEG_MOMENT_COMBO = {          # BDM 908.3 / 918.2.3.4, per MBE 6A.4.4.2.1
    "vehicle": "Type 3-3", "n_vehicles": 2, "vehicle_factor": 0.75,
    "lane_load_klf": 0.2, "headway_ft": 30.0,
}
LONG_SPAN_FT = 200.0          # 918.2.3.4 threshold
TRAIN_HEADWAY_FT = 30.0       # 918.2.3.x — 5C1 train spacing

def negative_moment_live_load(m_per_truck_kipft, span_ft):
    """Combination for negative moment / interior reactions."""
    c = NEG_MOMENT_COMBO
    return c["n_vehicles"] * c["vehicle_factor"] * m_per_truck_kipft \
        + c["lane_load_klf"] * span_ft ** 2 / 8.0

check("negative-moment vehicle factor", 0.75, NEG_MOMENT_COMBO["vehicle_factor"])
check("negative-moment lane load", 0.2, NEG_MOMENT_COMBO["lane_load_klf"], unit="klf")

[OK] negative-moment vehicle factor: guide=0.75 civilpy=0.75 
[OK] negative-moment lane load: guide=0.2 civilpy=0.2 klf


True

## 6. Permit load rating — MBE 6A.4.5

In [13]:
# TODO(guide): lrfr.permit_load_factor(...) — confirm permit type,
# escort conditions, and ADTT band against the guide's example.
pass

## 7. Posting and Midas rating-table cross-check

`lrfr.posting_load` for any legal RF < 1.0, then diff the full rating table
against what Civil NX reports.

In [14]:
# TODO(midas-api): "Load Rating Result" is a PSC *design-module* table.
# Verified 2026-07-27: the design result tables (fps, c, Mcr, Av,req,
# FDL/AFDL columns) are NOT exposed through the known /post/TABLE surface —
# they need the PSC Design run configured in the Civil NX UI (design code,
# PSC design parameters, Section Manager rebar) and/or the official JSON
# manual's design TABLE_TYPE names. Analysis-side tables ARE verified:
# BEAMFORCE, BEAMSTRESSPSC (the ten-check-point stress table with
# Sig-Is(shear), Sig-Is(shear+torsion), Sig-Ps(Max/Min) columns), REACTIONG.
print("PENDING: Load Rating Result — needs the PSC Design module (see comment)")

PENDING: Load Rating Result — needs the PSC Design module (see comment)


## Validation summary

Every `check()` recorded above, in one table. `PENDING` rows are waiting on
either guide values (hand entry) or the Midas API coming back online.

In [15]:
summary()

20/20 checks OK, 0 mismatches, 0 pending


,check,guide,civilpy,rel diff,unit,status
0,2F1 GVW,30.00,30.00,0.0,kip,OK
1,3F1 GVW,46.00,46.00,0.0,kip,OK
2,5C1 GVW,80.00,80.00,0.0,kip,OK
3,Type 3 GVW,50.00,50.00,0.0,kip,OK
4,Type 3S2 GVW,72.00,72.00,0.0,kip,OK
5,Type 3-3 GVW,80.00,80.00,0.0,kip,OK
6,SU4 GVW,54.00,54.00,0.0,kip,OK
7,SU5 GVW,62.00,62.00,0.0,kip,OK
8,SU6 GVW,69.50,69.50,0.0,kip,OK
9,SU7 GVW,77.50,77.50,0.0,kip,OK
